# The Complete Agent
### Pune AI Builders Meetup — Notebook 05

---

> **Everything so far:**
> - NB01 — LLM parsed the resume → structured profile (but with gaps and missing fields)
> - NB02 — LLM asked clarifying questions → complete profile
> - NB03 — LLM called tools → GitHub verified, jobs found
> - NB04 — LLM drove the session with a goal → autonomous assessment
>
> **This notebook:** One agent. One goal. Runs the entire session from a raw parsed profile
> to a final recommendation — asking questions, verifying GitHub, searching jobs, in the right order.

No human in the loop except at start and end.

---
## The Problem the Previous Notebooks Left Open

**NB04** had a complete profile handed to the agent — salary, notice period, gaps already explained.
In reality, after parsing a resume (NB01) you have a **partial** profile:

```
After NB01 parsing:
  ✓  name, role, skills, experience, projects, timeline
  ✗  salary expectation   ← not in resume
  ✗  notice period        ← not in resume  
  ✗  reason for leaving   ← not in resume
  ✗  relocation preference← not in resume
  ✗  gap explanations     ← gaps detected but unexplained
  ✗  github verified      ← claimed but not checked
```

A real career counsellor would **not** recommend jobs before knowing these.
The complete agent must:
1. Ask clarification questions until all gaps are filled
2. Verify GitHub
3. **Only then** search for jobs
4. Produce final recommendation

In [1]:
import os, json, requests
import pandas as pd
from openai import OpenAI

OPENAI_API_KEY = "your-api-key-here"  # ← replace with your key, ignoe if its in env variable
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY))

JOB_DATABASE = pd.read_csv("../data/jobs.csv")

# ── Partial profile — result of NB01 resume parsing ────────────────────────
# This is what we actually have after parsing the resume.
# The agent must fill the rest through conversation.

ARJUN_PARTIAL = {
    "name"           : "Arjun Mehta",
    "current_role"   : "Senior SDE-II at Flipkart",
    "total_experience": 8,
    "top_skills"     : ["Java", "Go", "Kafka", "Redis", "distributed systems"],
    "career_goal"    : "Senior IC or Tech Lead in fintech/infra",
    "github"         : "arjun91",
    "career_gaps"    : [
        {"from": "Aug 2017", "to": "Mar 2018", "duration_months": 7,  "reason_given": False},
        {"from": "Feb 2021", "to": "Sep 2021", "duration_months": 7,  "reason_given": False},
    ],
    # Not in resume — agent must ask
    "salary_expectation" : None,
    "notice_period"      : None,
    "reason_for_change"  : None,
    "open_to_relocation" : None,
}

print("Setup done. Profile fields:")
for k, v in ARJUN_PARTIAL.items():
    status = "✓" if v is not None else "✗ MISSING"
    print(f"  {status}  {k}")

Setup done. Profile fields:
  ✓  name
  ✓  current_role
  ✓  total_experience
  ✓  top_skills
  ✓  career_goal
  ✓  github
  ✓  career_gaps
  ✗ MISSING  salary_expectation
  ✗ MISSING  notice_period
  ✗ MISSING  reason_for_change
  ✗ MISSING  open_to_relocation


---
## The Agent's Tools

Four tools — one for each phase:

| Tool | Phase | Purpose |
|---|---|---|
| `ask_candidate` | Discovery | Ask one clarification question at a time |
| `read_github` | Verification | Check GitHub — real API, live data |
| `search_jobs` | Job Search | Match jobs from CSV — only after discovery |
| `finish_session` | Assessment | Produce structured final output — signals completion |

In [2]:
def ask_candidate(question: str) -> dict:
    """Ask the candidate a clarification question. Reads the answer from stdin."""
    print(f"\n  Counsellor : {question}")
    answer = input("  You        : ").strip() or "No answer."
    return {"question": question, "answer": answer}


# ── Tool implementations ───────────────────────────────────────────────────

def read_github(username: str) -> dict:
    """Fetch real GitHub profile via public API. Gracefully handles rate-limits."""
    demo_username = "mahadevTW"
    try:
        user  = requests.get(f"https://api.github.com/users/{demo_username}", timeout=5).json()
        repos = requests.get(f"https://api.github.com/users/{demo_username}/repos?sort=stars&per_page=10", timeout=5).json()
    except Exception as e:
        return {"error": str(e)}

    if not isinstance(user, dict) or "login" not in user:
        return {"error": user.get("message", "GitHub API unavailable") if isinstance(user, dict) else "GitHub API unavailable"}
    if not isinstance(repos, list):
        return {"error": repos.get("message", "Could not fetch repos") if isinstance(repos, dict) else "Could not fetch repos"}

    lang_count = {}
    for r in repos:
        if r.get("language"):
            lang_count[r["language"]] = lang_count.get(r["language"], 0) + 1
    return {
        "username"     : demo_username,
        "name"         : user.get("name"),
        "company"      : user.get("company"),
        "public_repos" : user.get("public_repos"),
        "followers"    : user.get("followers"),
        "top_languages": sorted(lang_count, key=lang_count.get, reverse=True)[:3],
        "top_repos"    : [{"name": r["name"], "stars": r["stargazers_count"],
                           "lang": r["language"], "desc": r["description"],
                           "last_pushed": r["pushed_at"][:10]} for r in repos[:5]],
    }

def search_jobs(skills: list, domain: str = None, min_years: int = 0) -> list:
    """Search job database by matching candidate skills against job descriptions."""
    results = []
    active_jobs = JOB_DATABASE[JOB_DATABASE["is_active"] == True]

    for _, row in active_jobs.iterrows():
        searchable = " ".join([
            str(row.get("job_title", "") or ""),
            str(row.get("job_description", "") or ""),
        ]).lower()

        matched = [s for s in skills if s.strip().lower() in searchable]
        overlap  = len(matched) / len(skills) if skills else 0

        if overlap >= 0.4:
            results.append({
                "id"            : row["id"],
                "job_uid"       : row["job_uid"],
                "job_title"     : row["job_title"],
                "company"       : row["company_name"],
                "location"      : row["location"],
                "salary"        : row.get("salary") or "Not disclosed",
                "matched_skills": matched,
                "match_score"   : round(overlap * 100),
                "apply_link"    : row["apply_link"],
            })

    return sorted(results, key=lambda x: x["match_score"], reverse=True)[:3]

def finish_session(recommendation: str, top_jobs: list, next_steps: list) -> dict:
    """Agent calls this when the full assessment is complete."""
    return {"status": "completed", "recommendation": recommendation,
            "top_jobs": top_jobs, "next_steps": next_steps}


# ── Tool dispatcher ────────────────────────────────────────────────────────

def execute_tool(name: str, args: dict):
    if name == "ask_candidate" : return ask_candidate(**args)
    if name == "read_github"   : return read_github(**args)
    if name == "search_jobs"   : return search_jobs(**args)
    if name == "finish_session": return finish_session(**args)
    raise ValueError(f"Unknown tool: {name}")


# ── Tool schema ────────────────────────────────────────────────────────────

AGENT_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "ask_candidate",
            "description": "Ask the candidate one clarification question. Use this to fill missing profile fields and explain career gaps. Ask one question at a time.",
            "parameters": {"type": "object",
                "properties": {"question": {"type": "string", "description": "A natural, conversational question to ask the candidate"}},
                "required": ["question"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_github",
            "description": "Fetch the candidate's GitHub profile to verify their technical claims — repos, languages, activity.",
            "parameters": {"type": "object",
                "properties": {"username": {"type": "string"}},
                "required": ["username"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_jobs",
            "description": "Search for matching jobs. Call this ONLY after discovery questions are answered and GitHub is verified.",
            "parameters": {"type": "object",
                "properties": {
                    "skills"   : {"type": "array", "items": {"type": "string"}},
                    "domain"   : {"type": "string"},
                    "min_years": {"type": "integer"}
                },
                "required": ["skills", "min_years"]}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "finish_session",
            "description": "Call this when ALL phases are done: questions answered, GitHub verified, jobs found.",
            "parameters": {"type": "object",
                "properties": {
                    "recommendation": {"type": "string", "description": "2-3 sentence career assessment grounded in real data"},
                    "top_jobs"      : {"type": "array",  "items": {"type": "string"},
                                       "description": "Copy the exact job_title and company from search_jobs results. Format: 'job_title at company (location)'. Never invent company names."},
                    "next_steps"    : {"type": "array",  "items": {"type": "string"}, "description": "3 concrete actions for the candidate"}
                },
                "required": ["recommendation", "top_jobs", "next_steps"]}
        }
    },
]

print(f"Tools registered: {[t['function']['name'] for t in AGENT_TOOLS]}")

Tools registered: ['ask_candidate', 'read_github', 'search_jobs', 'finish_session']


---
## The Agent Goal — Four Phases, Strict Order

The system prompt defines the phases and enforces the order.
The LLM decides how many questions to ask, when it has enough, and when to move on.

In [3]:
AGENT_GOAL = """
You are an AI career counsellor agent. Your goal: produce a complete, grounded career assessment.

Work through FOUR phases in strict order. Do NOT skip ahead.

──────────────────────────────────────────────────────
PHASE 1 — DISCOVERY  (use ask_candidate)
──────────────────────────────────────────────────────
Fill every missing profile field before moving on:
  • salary expectation
  • notice period
  • reason for leaving current role
  • open to relocation?
  • explanation for each career gap (if any)

Rules:
  - Ask ONE question at a time.
  - Prioritise unexplained career gaps first — they are red flags without context.
  - Stop asking when all fields above are known.

──────────────────────────────────────────────────────
PHASE 2 — VERIFICATION  (use read_github)
──────────────────────────────────────────────────────
Check the candidate's GitHub. Verify whether their claimed skills and projects
are visible in their public repos, this should only be done if github username is provided in the resume.

──────────────────────────────────────────────────────
PHASE 3 — JOB SEARCH  (use search_jobs)
──────────────────────────────────────────────────────
Only after phases 1 and 2 are complete.
Use the candidate's verified skills and experience to find the best matches

──────────────────────────────────────────────────────
PHASE 4 — FINISH  (use finish_session)
──────────────────────────────────────────────────────
Write your recommendation using real data from all three phases.
Reference actual answers, actual repos, actual job titles.
"""

print("Agent goal defined — 4 phases, strict order.")

Agent goal defined — 4 phases, strict order.


---
## The Agent Loop

Same loop as NB04 — but now the agent has `ask_candidate` to drive discovery
before it can use `search_jobs`.

```
start
  │
  ├─ ask_candidate (repeat until all gaps filled)
  │       └─ candidate answers → appended to context
  │
  ├─ read_github
  │       └─ live GitHub data → appended to context
  │
  ├─ search_jobs
  │       └─ job matches → appended to context
  │
  └─ finish_session  ──►  structured output
```

In [4]:
PHASE_LABELS = {
    "ask_candidate" : "PHASE 1 — Discovery",
    "read_github"   : "PHASE 2 — Verification",
    "search_jobs"   : "PHASE 3 — Job Search",
    "finish_session": "PHASE 4 — Finish",
}

def run_complete_agent(profile: dict) -> dict:
    messages = [
        {"role": "system", "content": AGENT_GOAL},
        {"role": "user",   "content": f"Candidate profile (parsed from resume):\n{json.dumps(profile, indent=2)}\n\nBegin the assessment."}
    ]

    current_phase = None

    for turn in range(1, 20):
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=AGENT_TOOLS
        )
        msg = resp.choices[0].message
        messages.append(msg)

        if not msg.tool_calls:
            print(f"Turn {turn}: Agent stopped unexpectedly.")
            return {}

        for tc in msg.tool_calls:
            args   = json.loads(tc.function.arguments)
            result = execute_tool(tc.function.name, args)

            # Print phase header when phase changes
            label = PHASE_LABELS.get(tc.function.name)
            if label != current_phase:
                current_phase = label
                print(f"\n{'─'*50}")
                print(f"  {label}")
                print(f"{'─'*50}")

            # Print what the agent did
            if tc.function.name == "ask_candidate":
                print(f"  Q: {args['question']}")
                print(f"  A: {result['answer']}")
            elif tc.function.name == "read_github":
                if "error" in result:
                    print(f"  GitHub: unavailable ({result['error']}) — skipping")
                else:
                    print(f"  GitHub: {result.get('name')} | repos: {result.get('public_repos')} | languages: {result.get('top_languages')}")
            elif tc.function.name == "search_jobs":
                print(f"  Found {len(result)} matches for skills: {args['skills'][:3]}...")
            elif tc.function.name == "finish_session":
                print(f"\n  Agent completed in {turn} turns.")
                return result

            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})

    return {}

print("run_complete_agent() ready.")


run_complete_agent() ready.


---
## Demo — The Complete Session

In [5]:
print("=" * 50)
print("COMPLETE AGENT SESSION")
print("=" * 50)

output = run_complete_agent(ARJUN_PARTIAL)

print()
print("=" * 50)
print("FINAL OUTPUT")
print("=" * 50)
print(f"\nAssessment:")
print(f"  {output.get('recommendation')}")
print(f"\nTop Jobs:")
for job in output.get("top_jobs", []):
    print(f"  • {job}")
print(f"\nNext Steps:")
for step in output.get("next_steps", []):
    print(f"  → {step}")

COMPLETE AGENT SESSION

  Counsellor : Can you please explain the career gap from August 2017 to March 2018?

──────────────────────────────────────────────────
  PHASE 1 — Discovery
──────────────────────────────────────────────────
  Q: Can you please explain the career gap from August 2017 to March 2018?
  A: meidcl;e issue

  Counsellor : Thank you for that. Can you also explain the career gap from February 2021 to September 2021?
  Q: Thank you for that. Can you also explain the career gap from February 2021 to September 2021?
  A: same

  Counsellor : What are your salary expectations for your next role?
  Q: What are your salary expectations for your next role?
  A: 80LPA

  Counsellor : What is your current notice period?
  Q: What is your current notice period?
  A: 2 months

  Counsellor : What is your reason for leaving your current role at Flipkart?
  Q: What is your reason for leaving your current role at Flipkart?
  A: better job

  Counsellor : Are you open to relocation

---
## The Aha Moment

The agent asked questions **in the right order** — unexplained gaps before salary,
because gaps are red flags without context. It didn't search for jobs until it had
salary, notice, and a verified GitHub. Nobody told it to do this — it followed the goal.

Every notebook contributed one piece:

| Notebook | Capability | Contributed to this agent |
|---|---|---|
| NB01 | **Understanding** | Resume parsed → partial profile as starting point |
| NB02 | **Inference** | `ask_candidate` loop — one question at a time, stops when ready |
| NB03 | **Tool Calling** | `read_github` (live API) + `search_jobs` (CSV) |
| NB04 | **Agent Loop** | Goal-driven loop, `finish_session` as structured exit |
| NB05 | **Complete Agent** | All four wired together with phase ordering |

---

```
Intelligence is not in the LLM.
Intelligence is in the system:

  Goal  +  Tools  +  Memory  +  Loop  =  Agent
   NB04      NB03      NB01       NB04    NB05
```

**GPT is one component. You design the rest.**

---

> **Understand → Infer → Act → Agent Loop → *Complete Agent***